# Pelatihan DINO-DETR: Deteksi Pemain & Bola (Fine-Tuning Baseline)

Notebook ini mendemonstrasikan secara interaktif seluruh alur kerja proyek deteksi objek **DINO-DETR** dalam presisi **FP32 murni** dengan memuat **bobot praterlatih (pretrained COCO weights)** untuk melakukan **Fine-Tuning** guna mendeteksi dua objek: pemain sepak bola (`person`) dan bola (`ball`).

---

## 1. Pemasangan Pustaka & Kloning Repositori DINO Resmi

Langkah pertama adalah memasang semua dependensi python dari `requirements.txt` dan mengkloning repositori DINO resmi dari IDEA-Research.

In [ ]:
# Install requirements
!pip install -r requirements.txt

# Clone DINO official repository if it does not exist
import os
if not os.path.exists("official_dino"):
    !git clone https://github.com/IDEA-Research/DINO.git official_dino
else:
    print("[INFO] official_dino already cloned.")

### Kompilasi C++/CUDA Operators (Ops)

Kompilasi modul *Multi-Scale Deformable Attention* C++/CUDA bawaan DINO. Jika kompilasi ini gagal karena ketiadaan compiler C++ di Windows atau ketidakcocokan versi CUDA, model kita akan secara otomatis menggunakan **Pure PyTorch Fallback (`F.grid_sample`)** yang didefinisikan di `model.py`.

In [ ]:
# Attempt compilation
%cd official_dino/models/dino/ops
!python setup.py build develop
%cd ../../../../

## 2. Konversi Dataset YOLO ke COCO JSON

Kita memproses dataset di folder `./merged_yolo_person_ball` untuk menghasilkan anotasi standar COCO JSON (`annotations_train.json` dan `annotations_val.json`).

In [ ]:
!python convert_yolo_to_coco.py

## 3. Visualisasi Gambar Sanity-Check

Sebelum melatih model, mari kita tampilkan salah satu gambar hasil sanity-check di folder `./sanity_checks` untuk memastikan letak kotak bounding box sudah akurat di atas objek gambar.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob

sanity_images = glob.glob("./sanity_checks/*.png")
if sanity_images:
    img = Image.open(sanity_images[0])
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("[WARNING] Gambar sanity check belum dibuat. Pastikan langkah konversi telah dijalankan dengan sukses.")

## 4. Dataset Loader & Custom Augmentations

Kita menguji performa `FootballDataset` dan modul `DETRTransforms` yang mengonversi koordinat boks dan menormalisasi gambar menggunakan rata-rata ImageNet.

In [ ]:
from dataset import FootballDataset, DETRTransforms, collate_fn
from torch.utils.data import DataLoader

# Inisialisasi dataset validasi
val_dataset = FootballDataset(
    annotations_path="annotations_val.json",
    images_dir="./merged_yolo_person_ball/images/val",
    transforms=DETRTransforms(is_train=False)
)

# Buat sample loader
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
images, targets = next(iter(val_loader))

print(f"Loaded batch image size : {images[0].shape}")
print(f"Target boxes format     : {targets[0]['boxes']}")

## 5. Inisialisasi Model DINO-DETR (Fine-Tuning)

Kita memuat arsitektur DINO. Proses pemuatan weights COCO pretrained dilakukan sebelum linear classifier head diganti menjadi tepat **`num_classes = 2`**. 

Bobot pretrained resmi `dino_r50_4scale_12ep.pth` akan otomatis diunduh jika belum ada di direktori lokal Anda.

In [ ]:
from model import build_dino_model
from train import download_pretrained_weights
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Unduh bobot pretrained jika belum ada
checkpoint_name = "dino_r50_4scale_12ep.pth"
download_pretrained_weights(checkpoint_name)

# Bangun model dan muat bobot pretrained secara aman sebelum memotong head klasifikasi
model = build_dino_model(num_classes=2, checkpoint_path=checkpoint_name, device=device)
print(model)

## 6. Uji Coba Overfitting Sanity Check (10 Gambar)

Melatih model pada subset kecil (10 gambar) selama 50 epoch menggunakan bobot pretrained ini. Karena model sudah terlatih dengan bobot COCO, loss latihan akan turun drastis ke arah nol.

In [ ]:
from train import run_overfit_check

# Jalankan tes overfit
run_overfit_check(model, device)

## 7. Eksekusi Pelatihan Fine-Tuning Penuh (FP32 Baseline)

Jalankan training fine-tuning baseline penuh selama 12 epoch dengan parameter batch size yang disesuaikan kapasitas VRAM GPU Anda. Jika terjadi OOM VRAM, skrip akan menurunkan batch size secara dinamis untuk mencegah crash.

In [ ]:
from train import train_baseline

class Args:
    def __init__(self):
        self.epochs = 12
        self.batch_size = 8 # Sesuaikan dengan kapasitas VRAM GPU (misal 8 atau 16)
        self.lr = 1e-4
        self.weight_decay = 1e-4
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.checkpoint = "dino_r50_4scale_12ep.pth" # Secara otomatis didownload jika belum ada
        self.overfit_check = False
        self.log_interval = 50

args = Args()
train_baseline(args)